In [ ]:
import os
import sys
sys.dont_write_bytecode = True

from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv())

import warnings
warnings.filterwarnings("ignore")

import logging
import logging.config
from transformers import logging as hf_logging

hf_logging.set_verbosity_error()

logging.config.dictConfig({
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {
        "default": {
            "format": "%(asctime)s [%(name)s] %(levelname)s  %(message)s",
            "datefmt": "%H:%M:%S",
        }
    },
    "handlers": {
        "console": {
            "class": "logging.StreamHandler",
            "formatter": "default",
        }
    },
    "root": {
        "level": "INFO",
        "handlers": ["console"],
    },
    "loggers": {
        "transformers": {"level": "ERROR", "propagate": False},
        "rouge":        {"level": "ERROR", "propagate": False},
        "bert_score":   {"level": "ERROR", "propagate": False},
    },
})


# api keys
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
BASE_URL = os.getenv("BASE_URL")

# model selection
SMALL_MODEL_NAME = "mistralai/ministral-8b-2512"
LARGE_MODEL_NAME = "qwen/qwen3.5-35b-a3b"
JUDGE_MODEL_NAME = "qwen/qwen3.5-122b-a10b"
SLM_AS_ROUTER = "mistralai/ministral-3b-2512"

# concurrency level
RPS = 5

# additional configuration
DATA_LOCATION_PATH = "generated/rag_bench_techqa/data.parquet"
TOKENIZER_NAME = "o200k_base"
SPACY_NLP_MODEL = "en_core_web_lg"

# policies
SLM_AS_ROUTER_CONFIDENCE_THRESHOLD = 4

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

from core.data import RAGBenchDataset
from core.pipeline import RAGPipelineRunner, JScore, InferenceRecord
from core.messaging import LangchainMessageBuilder, PROMPT_REGISTRY
from core.router import (
    SLMRouterOutput,
    SLMRoutingPolicy,
    WeightedRuleBasedRoutingPolicy,
    WeightedRule
)
from core.tasks import RAGTaskPrediction
from core.utils import (
    get_evaluation_summary,
    compute_slm_routing_metrics,
    dump_to_csv
)

In [ ]:
dataset = RAGBenchDataset.from_files(DATA_LOCATION_PATH)

In [ ]:
dataset[:1]

In [ ]:
dataset = dataset[:100]

In [ ]:
_base_rules = [
    WeightedRule(
        name="query_token_count",
        operator="ge",
        threshold=55,
        weight=0.10,
    ),
    WeightedRule(
        name="query_noun_chunk_count",
        operator="ge",
        threshold=7,
        weight=0.15,
    ),
    WeightedRule(
        name="query_avg_word_frequency",
        operator="le",
        threshold=3.0,
        weight=0.20,
    ),
    WeightedRule(
        name="avg_lexical_overlap",
        operator="le",
        threshold=0.10,
        weight=0.25,
    ),
]

qa_task_rules = _base_rules + [
    WeightedRule(
        name="max_lexical_overlap",
        operator="le",
        threshold=0.15,
        weight=0.20,
    ),
    WeightedRule(
        name="relevant_documents_ratio",
        operator="le",
        threshold=0.20,
        weight=0.25,
    ),
    WeightedRule(
        name="max_semantic_similarity",
        operator="le",
        threshold=0.80,
        weight=0.20,
    ),
    WeightedRule(
        name="documents_count",
        operator="ge",
        threshold=5,
        weight=0.10,
    )
]

message_builder = LangchainMessageBuilder.from_sequence(
    ("question_answering", PROMPT_REGISTRY.question_answering_inference, RAGTaskPrediction),
    ("judge", PROMPT_REGISTRY.evaluation, JScore)
)
model_kwargs = {
    "api_key": OPENROUTER_API_KEY,
    "base_url": BASE_URL,
    "max_tokens": 4092,
    "temperature": 0.0,
    "rate_limiter": InMemoryRateLimiter(
        requests_per_second=RPS,
        check_every_n_seconds=0.1,
        max_bucket_size=RPS * 2
    )
}

### No routing: SLM

In [ ]:
ragbench_slm_only_pipeline = RAGPipelineRunner(
    small_model=SMALL_MODEL_NAME,
    large_model=LARGE_MODEL_NAME,
    judge_model=JUDGE_MODEL_NAME,
    routing_mode="slm",
    messages_builder=message_builder,
    extractor_spacy_nlp=SPACY_NLP_MODEL,
    extractor_tokenizer_name=TOKENIZER_NAME,
    model_kwargs=model_kwargs
)

In [ ]:
ragbench_slm_only_pipeline_result = await ragbench_slm_only_pipeline.arun(dataset)
dump_to_csv(ragbench_slm_only_pipeline_result, path="ragbench_slm_only_pipeline_result")

In [ ]:
ragbench_slm_only_pipeline_evaluation = await ragbench_slm_only_pipeline.aevaluate(ragbench_slm_only_pipeline_result)
dump_to_csv(ragbench_slm_only_pipeline_evaluation, path="ragbench_slm_only_pipeline_evaluation")

In [ ]:
ragbench_slm_only_pipeline_summary = get_evaluation_summary(ragbench_slm_only_pipeline_evaluation)
print(ragbench_slm_only_pipeline_summary)

### No routing: LLM

In [ ]:
ragbench_llm_only_pipeline = RAGPipelineRunner(
    small_model=SMALL_MODEL_NAME,
    large_model=LARGE_MODEL_NAME,
    judge_model=JUDGE_MODEL_NAME,
    routing_mode="llm",
    messages_builder=message_builder,
    extractor_spacy_nlp=SPACY_NLP_MODEL,
    extractor_tokenizer_name=TOKENIZER_NAME,
    model_kwargs=model_kwargs
)

In [ ]:
ragbench_llm_only_pipeline_result = await ragbench_llm_only_pipeline.arun(dataset)
dump_to_csv(ragbench_llm_only_pipeline_result, path="ragbench_llm_only_pipeline_result")

In [ ]:
ragbench_llm_only_pipeline_evaluation = await ragbench_llm_only_pipeline.aevaluate(ragbench_llm_only_pipeline_result)
dump_to_csv(ragbench_llm_only_pipeline_evaluation, path="ragbench_llm_only_pipeline_evaluation")

In [ ]:
ragbench_llm_only_pipeline_summary = get_evaluation_summary(ragbench_llm_only_pipeline_evaluation)
print(ragbench_llm_only_pipeline_summary)

### Dynamic routing: Rule-based, ministral-8b-2512

In [ ]:
_common_rule_based_kwargs = {
    "routing_mode": "dynamic",
    "messages_builder": message_builder,
    "dynamic_routing_policies": {
        "question_answering": WeightedRuleBasedRoutingPolicy(
            *qa_task_rules,
            min_triggers=3,
            cumulative_weights_threshold=0.65
        )
    },
    "extractor_spacy_nlp": SPACY_NLP_MODEL,
    "extractor_tokenizer_name": TOKENIZER_NAME,
    "model_kwargs": model_kwargs
}

In [ ]:
ragbench_wrb_pipeline = RAGPipelineRunner(
    small_model=SMALL_MODEL_NAME,
    large_model=LARGE_MODEL_NAME,
    judge_model=JUDGE_MODEL_NAME,
    **_common_rule_based_kwargs
)

In [ ]:
ragbench_wrb_pipeline_result = await ragbench_wrb_pipeline.arun(dataset)
dump_to_csv(ragbench_wrb_pipeline_result, path="ragbench_wrb_pipeline_result")

In [ ]:
ragbench_wrb_pipeline_evaluation = await ragbench_wrb_pipeline.aevaluate(ragbench_wrb_pipeline_result)
dump_to_csv(ragbench_wrb_pipeline_evaluation, path="ragbench_wrb_pipeline_evaluation")

In [ ]:
rule_based__ministral_8b_summary = get_evaluation_summary(ragbench_wrb_pipeline_evaluation)
print(rule_based__ministral_8b_summary)

In [ ]:
ragbench_wrb_pipeline_routing_metrics = compute_slm_routing_metrics(ragbench_wrb_pipeline_evaluation)
dump_to_csv([ragbench_wrb_pipeline_routing_metrics], path="rule_based_pipeline_routing_metrics")
print(ragbench_wrb_pipeline_routing_metrics)

### Dynamic routing: SLM as a router (ministral-3b-2512)

In [ ]:
slm_router_client = ChatOpenAI(model=SLM_AS_ROUTER, **model_kwargs)

slm_routing_policy_messages_builder = LangchainMessageBuilder.from_sequence(
    ("slm_routing_policy", PROMPT_REGISTRY.slm_as_router, SLMRouterOutput)
)

slm_as_router_policy = SLMRoutingPolicy(
    client=slm_router_client,
    message_builder=slm_routing_policy_messages_builder,
    confidence_threshold=SLM_AS_ROUTER_CONFIDENCE_THRESHOLD
)

In [ ]:
_common_slm_based_kwargs = {
    "routing_mode": "dynamic",
    "messages_builder": message_builder,
    "dynamic_routing_policies": {
        "question_answering": slm_as_router_policy
    },
    "extractor_spacy_nlp": SPACY_NLP_MODEL,
    "extractor_tokenizer_name": TOKENIZER_NAME,
    "model_kwargs": model_kwargs
}

In [ ]:
ragbench_slmr_pipeline = RAGPipelineRunner(
    small_model=SMALL_MODEL_NAME,
    large_model=LARGE_MODEL_NAME,
    judge_model=JUDGE_MODEL_NAME,
    **_common_slm_based_kwargs
)

In [ ]:
ragbench_slmr_pipeline_result = await ragbench_slmr_pipeline.arun(dataset)
dump_to_csv(ragbench_slmr_pipeline_result, path="ragbench_slmr_pipeline_result")

In [ ]:
ragbench_slmr_pipeline_evaluation = (
    await ragbench_slmr_pipeline.aevaluate(ragbench_slmr_pipeline_result)
)
dump_to_csv(ragbench_slmr_pipeline_evaluation, path="ragbench_slmr_pipeline_evaluation")

In [ ]:
ragbench_slmr_pipeline_summary = get_evaluation_summary(ragbench_slmr_pipeline_evaluation)
print(ragbench_slmr_pipeline_summary)

In [ ]:
ragbench_slmr_pipeline_routing_metrics = compute_slm_routing_metrics(ragbench_slmr_pipeline_evaluation)
dump_to_csv([ragbench_slmr_pipeline_routing_metrics], path="ragbench_slmr_pipeline_routing_metrics")
print(ragbench_slmr_pipeline_routing_metrics)